# CREATE or REPLACE materialized lake views 
1. Use this notebook to CREATE or REPLACE materialized lake views. 
2. Select **Run all** to run the notebook. 
3. When the notebook run is completed, return to your lakehouse and refresh your materialized lake views graph. 

**Make sure you connect the correct Lakehouse from on your Data Workspace as default lakehouse**


In [ ]:
CREATE SCHEMA IF NOT EXISTS gold

In [ ]:
-- ============================================================================
-- 1. Helper MLV: latest inventory per set
--    Encapsulates the "MAX(version) per set_num" rule so every fact view
--    just joins to this and gets a single inventory row per set.
-- ============================================================================

CREATE or REPLACE MATERIALIZED LAKE VIEW gold.vw_LatestInventory 
AS
SELECT
    inv.id          AS Inventory_ID,
    inv.set_num     AS Set_Num,
    inv.version     AS Inventory_Version
FROM LH_GOLD_LAYER.dbo.inventories AS inv
INNER JOIN (
    SELECT set_num, MAX(version) AS max_version
    FROM   LH_GOLD_LAYER.dbo.inventories
    GROUP BY set_num
) AS latest
    ON  latest.set_num     = inv.set_num
    AND latest.max_version = inv.version;


# DIMENSIONS

In [ ]:
-- ----------------------------------------------------------------------------
-- Dim_Theme
--   Themes self-reference via parent_id. Rebrickable's tree is shallow
--   (<=4 levels), so we resolve depth, root and the flattened L1..L4 path
--   with simple self-joins -- no recursive CTE required.
--   If a future theme is deeper than 4, it will still appear correctly,
--   but the leaf name will only be visible via Theme_Name / Theme_Path
--   (not via Theme_Level_1..4).
-- ----------------------------------------------------------------------------

CREATE OR REPLACE MATERIALIZED LAKE VIEW LH_GOLD_LAYER.gold.Dim_Theme
AS
SELECT
    t.id                                          AS Theme_ID,
    t.name                                        AS Theme_Name,
    t.parent_id                                   AS Parent_Theme_ID,

    -- Depth: 1 if root, else 1 + depth of parent (capped at 5+ for slicing).
    CASE
        WHEN t.parent_id  IS NULL THEN 1
        WHEN p1.parent_id IS NULL THEN 2
        WHEN p2.parent_id IS NULL THEN 3
        WHEN p3.parent_id IS NULL THEN 4
        ELSE 5
    END                                           AS Theme_Depth,

    -- Root: the deepest non-null ancestor (or self if t is already root).
    COALESCE(p3.id,   p2.id,   p1.id,   t.id)     AS Root_Theme_ID,
    COALESCE(p3.name, p2.name, p1.name, t.name)   AS Root_Theme_Name,

    -- Human-readable path from root to leaf.
    CONCAT_WS(' / ',
        COALESCE(p3.name, p2.name, p1.name, t.name),                       -- L1 (root)
        CASE WHEN p3.name IS NOT NULL THEN p2.name
             WHEN p2.name IS NOT NULL THEN p1.name
             WHEN p1.name IS NOT NULL THEN t.name
        END,                                                                -- L2
        CASE WHEN p3.name IS NOT NULL THEN p1.name
             WHEN p2.name IS NOT NULL THEN t.name
        END,                                                                -- L3
        CASE WHEN p3.name IS NOT NULL THEN t.name END                       -- L4
    )                                             AS Theme_Path,

    -- Flattened levels: Level_N is the ancestor at depth N (NULL if shallower).
    COALESCE(p3.name, p2.name, p1.name, t.name)   AS Theme_Level_1,

    CASE
        WHEN p3.name IS NOT NULL THEN p2.name      -- t at depth 4 -> L2 = p2
        WHEN p2.name IS NOT NULL THEN p1.name      -- t at depth 3 -> L2 = p1
        WHEN p1.name IS NOT NULL THEN t.name       -- t at depth 2 -> L2 = t
    END                                           AS Theme_Level_2,

    CASE
        WHEN p3.name IS NOT NULL THEN p1.name      -- t at depth 4 -> L3 = p1
        WHEN p2.name IS NOT NULL THEN t.name       -- t at depth 3 -> L3 = t
    END                                           AS Theme_Level_3,

    CASE
        WHEN p3.name IS NOT NULL THEN t.name       -- t at depth 4 -> L4 = t
    END                                           AS Theme_Level_4

FROM       LH_GOLD_LAYER.dbo.themes AS t
LEFT JOIN  LH_GOLD_LAYER.dbo.themes AS p1 ON p1.id = t.parent_id
LEFT JOIN  LH_GOLD_LAYER.dbo.themes AS p2 ON p2.id = p1.parent_id
LEFT JOIN  LH_GOLD_LAYER.dbo.themes AS p3 ON p3.id = p2.parent_id;


In [ ]:
-- ----------------------------------------------------------------------------
-- Dim_Color
--   Adds a synthetic "Unknown" member for color_id = -1 (used by some
--   inventory_parts rows as a placeholder) when the source colors table
--   doesn't already include it.
-- ----------------------------------------------------------------------------
CREATE OR REPLACE MATERIALIZED LAKE VIEW LH_GOLD_LAYER.gold.Dim_Color
AS
SELECT
    c.id                            AS Color_ID,
    c.name                          AS Color_Name,
    c.rgb                           AS Color_RGB,
    CASE
        WHEN LOWER(CAST(c.is_trans AS STRING)) IN ('t','true','1') THEN TRUE
        ELSE FALSE
    END                             AS Is_Transparent
FROM LH_GOLD_LAYER.dbo.colors AS c

UNION ALL

SELECT
    -1               AS Color_ID,
    'Unknown'        AS Color_Name,
    '000000'         AS Color_RGB,
    FALSE            AS Is_Transparent
FROM (SELECT 1 AS dummy) AS d
WHERE NOT EXISTS (
    SELECT 1 FROM LH_GOLD_LAYER.dbo.colors WHERE id = -1
);

In [ ]:
-- ----------------------------------------------------------------------------
-- Dim_PartCategory
-- ----------------------------------------------------------------------------
CREATE OR REPLACE MATERIALIZED LAKE VIEW LH_GOLD_LAYER.gold.Dim_PartCategory
AS
SELECT
    pc.id     AS PartCategory_ID,
    pc.name   AS PartCategory_Name
FROM LH_GOLD_LAYER.dbo.part_categories AS pc;

In [ ]:
-- ----------------------------------------------------------------------------
-- Dim_Part
--   part_num is the natural key.
-- ----------------------------------------------------------------------------
CREATE OR REPLACE MATERIALIZED LAKE VIEW LH_GOLD_LAYER.gold.Dim_Part
AS
SELECT
    p.part_num            AS Part_ID,
    p.name                AS Part_Name,
    p.part_cat_id         AS PartCategory_ID,
    p.part_material       AS Part_Material
FROM LH_GOLD_LAYER.dbo.parts AS p;

In [ ]:
-- ----------------------------------------------------------------------------
-- Dim_Minifig
--   Minifigs share the inventory plumbing with sets but represent a different
--   business entity, so they get their own dimension.
-- ----------------------------------------------------------------------------
CREATE OR REPLACE MATERIALIZED LAKE VIEW LH_GOLD_LAYER.gold.Dim_Minifig
AS
SELECT
    m.fig_num     AS Minifig_ID,
    m.name        AS Minifig_Name,
    m.num_parts   AS Minifig_Num_Parts
FROM LH_GOLD_LAYER.dbo.minifigs AS m;

In [ ]:
-- ----------------------------------------------------------------------------
-- Dim_Set
-- ----------------------------------------------------------------------------
CREATE OR REPLACE MATERIALIZED LAKE VIEW LH_GOLD_LAYER.gold.Dim_Set
AS
SELECT
    s.set_num         AS Set_ID,
    s.name            AS Set_Name,
    s.year            AS Set_Year,
    s.year            AS Year_ID,        -- FK to Dim_Year
    s.theme_id        AS Theme_ID,       -- FK to Dim_Theme
    s.num_parts       AS Set_Num_Parts,  -- Rebrickable's own piece count (sanity check)
    s.img_url         AS Set_Image_URL
FROM LH_GOLD_LAYER.dbo.sets AS s;

In [ ]:
-- ----------------------------------------------------------------------------
-- Dim_Year
--   Built from the actual year range present in the sets table.
-- ----------------------------------------------------------------------------
CREATE OR REPLACE MATERIALIZED LAKE VIEW LH_GOLD_LAYER.gold.Dim_Year
AS
SELECT DISTINCT
    s.year                                                  AS Year_ID,
    s.year                                                  AS Calendar_Year,
    CAST(s.year / 10 AS INT) * 10                           AS Decade,
    CONCAT(CAST(CAST(s.year / 10 AS INT) * 10 AS STRING), 's') AS Decade_Label
FROM LH_GOLD_LAYER.dbo.sets AS s
WHERE s.year IS NOT NULL;

In [ ]:
-- ----------------------------------------------------------------------------
-- Dim_Year
--   Built from the actual year range present in the sets table.
-- ----------------------------------------------------------------------------
CREATE OR REPLACE MATERIALIZED LAKE VIEW LH_GOLD_LAYER.gold.Dim_Member
AS
SELECT Member_ID, Member_Name
FROM (
    VALUES
        (1001, 'Emmet Brickowski'),
        (1002, 'Wyldstyle Lucy'),
        (1003, 'Benny Spaceman'),
        (1004, 'Lord Business'),
        (1005, 'Kai Firefang'),
        (1006, 'Jay Walker'),
        (1007, 'Lloyd Garmadon'),
        (1008, 'Nya Waterfall'),
        (1009, 'Cole Brookstone'),
        (1010, 'Zane Julien'),
        (1011, 'Sensei Wu'),
        (1012, 'Laval Lionheart'),
        (1013, 'Eris Eagleton'),
        (1014, 'Chase McCain'),
        (1015, 'Rex Dangervest'),
        (1016, 'Unikitty Cloudcuckoo'),
        (1017, 'Metalbeard Pirate'),
        (1018, 'Clutch Powers')
) AS members(Member_ID, Member_Name)

# FACTS

In [ ]:
-- ----------------------------------------------------------------------------
-- Fact_Sets
--   Grain : one row per set.
--   Pre-aggregated measures from the latest inventory:
--     Set_Count             : always 1 (additive set count)
--     Total_Pieces          : sum of quantity (incl. spares)
--     Total_Pieces_NoSpare  : sum of quantity excluding spare parts
--     Unique_Parts          : distinct part_num (any color)
--     Unique_Part_Colors    : distinct part_num + color_id combinations
--     Total_Minifigs        : sum of minifig quantities in the set
-- ----------------------------------------------------------------------------
CREATE OR REPLACE MATERIALIZED LAKE VIEW LH_GOLD_LAYER.gold.Fact_Sets
AS
WITH inv_parts AS (
    SELECT
        li.Set_Num,
        SUM(CAST(ip.quantity AS INT))                                  AS Total_Pieces,
        SUM(CASE
                WHEN LOWER(CAST(ip.is_spare AS STRING)) IN ('t','true','1') THEN 0
                ELSE CAST(ip.quantity AS INT)
            END)                                                       AS Total_Pieces_NoSpare,
        COUNT(DISTINCT ip.part_num)                                    AS Unique_Parts,
        COUNT(DISTINCT CONCAT(ip.part_num, '|', CAST(ip.color_id AS STRING))) AS Unique_Part_Colors
    FROM LH_GOLD_LAYER.gold.vw_LatestInventory AS li
    INNER JOIN LH_GOLD_LAYER.dbo.inventory_parts AS ip
        ON ip.inventory_id = li.Inventory_ID
    GROUP BY li.Set_Num
),
inv_figs AS (
    SELECT
        li.Set_Num,
        SUM(CAST(im.quantity AS INT)) AS Total_Minifigs
    FROM LH_GOLD_LAYER.gold.vw_LatestInventory AS li
    INNER JOIN LH_GOLD_LAYER.dbo.inventory_minifigs AS im
        ON im.inventory_id = li.Inventory_ID
    GROUP BY li.Set_Num
)
SELECT
    s.set_num                                AS Set_ID,
    s.theme_id                               AS Theme_ID,
    s.year                                   AS Year_ID,
    1                                        AS Set_Count,             -- additive
    COALESCE(s.num_parts, 0)                 AS Set_Num_Parts_Source,  -- as declared in sets.csv
    COALESCE(ip.Total_Pieces, 0)             AS Total_Pieces,
    COALESCE(ip.Total_Pieces_NoSpare, 0)     AS Total_Pieces_NoSpare,
    COALESCE(ip.Unique_Parts, 0)             AS Unique_Parts,
    COALESCE(ip.Unique_Part_Colors, 0)       AS Unique_Part_Colors,
    COALESCE(f.Total_Minifigs, 0)            AS Total_Minifigs
FROM LH_GOLD_LAYER.dbo.sets AS s
LEFT JOIN inv_parts AS ip ON ip.Set_Num = s.set_num
LEFT JOIN inv_figs  AS f  ON f.Set_Num  = s.set_num;

In [ ]:
-- ----------------------------------------------------------------------------
-- Fact_SetParts
--   Grain : one row per (Set, Part, Color) on the latest inventory.
--   Use for piece-level analytics: most-used parts, rarest parts, parts by
--   theme/year, etc.
-- ----------------------------------------------------------------------------
CREATE OR REPLACE MATERIALIZED LAKE VIEW LH_GOLD_LAYER.gold.Fact_SetParts
AS
SELECT
    li.Set_Num                               AS Set_ID,
    s.theme_id                               AS Theme_ID,
    s.year                                   AS Year_ID,
    ip.part_num                              AS Part_ID,
    p.part_cat_id                            AS PartCategory_ID,
    ip.color_id                              AS Color_ID,
    CAST(ip.quantity AS INT)                 AS Quantity,
    CASE
        WHEN LOWER(CAST(ip.is_spare AS STRING)) IN ('t','true','1') THEN 1
        ELSE 0
    END                                      AS Is_Spare,
    CASE
        WHEN LOWER(CAST(ip.is_spare AS STRING)) IN ('t','true','1') THEN 0
        ELSE CAST(ip.quantity AS INT)
    END                                      AS Quantity_NoSpare
FROM LH_GOLD_LAYER.gold.vw_LatestInventory AS li
INNER JOIN LH_GOLD_LAYER.dbo.inventory_parts AS ip ON ip.inventory_id = li.Inventory_ID
INNER JOIN LH_GOLD_LAYER.dbo.sets            AS s  ON s.set_num       = li.Set_Num
LEFT  JOIN LH_GOLD_LAYER.dbo.parts           AS p  ON p.part_num      = ip.part_num;

In [ ]:
-- ----------------------------------------------------------------------------
-- Fact_SetMinifigs
--   Grain : one row per (Set, Minifig) on the latest inventory.
-- ----------------------------------------------------------------------------
CREATE OR REPLACE MATERIALIZED LAKE VIEW LH_GOLD_LAYER.gold.Fact_SetMinifigs
AS
SELECT
    li.Set_Num                  AS Set_ID,
    s.theme_id                  AS Theme_ID,
    s.year                      AS Year_ID,
    im.fig_num                  AS Minifig_ID,
    CAST(im.quantity AS INT)    AS Quantity
FROM LH_GOLD_LAYER.gold.vw_LatestInventory AS li
INNER JOIN LH_GOLD_LAYER.dbo.inventory_minifigs AS im ON im.inventory_id = li.Inventory_ID
INNER JOIN LH_GOLD_LAYER.dbo.sets               AS s  ON s.set_num       = li.Set_Num;

In [ ]:
-- ----------------------------------------------------------------------------
-- Bridge_MemberSet
-- ----------------------------------------------------------------------------
CREATE OR REPLACE MATERIALIZED LAKE VIEW LH_GOLD_LAYER.gold.Bridge_MemberSet
AS
SELECT Member_ID, Set_ID
FROM (
    VALUES
        -- Emmet Brickowski (1001) — 20 sets
        (1001, '75192-1'), (1001, '75290-1'), (1001, '10294-1'), (1001, '71043-1'),
        (1001, '10276-1'), (1001, '75313-1'), (1001, '21054-1'), (1001, '42115-1'),
        (1001, '10297-1'), (1001, '75309-1'), (1001, '10300-1'), (1001, '21330-1'),
        (1001, '10305-1'), (1001, '75341-1'), (1001, '42143-1'), (1001, '10312-1'),
        (1001, '75375-1'), (1001, '10321-1'), (1001, '21336-1'), (1001, '10307-1'),

        -- Wyldstyle Lucy (1002) — 15 sets
        (1002, '10255-1'), (1002, '21318-1'), (1002, '10270-1'), (1002, '75257-1'),
        (1002, '10281-1'), (1002, '10280-1'), (1002, '21327-1'), (1002, '10311-1'),
        (1002, '40460-1'), (1002, '10278-1'), (1002, '10264-1'), (1002, '21325-1'),
        (1002, '10290-1'), (1002, '10274-1'), (1002, '76218-1'),

        -- Benny Spaceman (1003) — 12 sets
        (1003, '10266-1'), (1003, '21321-1'), (1003, '10283-1'), (1003, '21309-1'),
        (1003, '75355-1'), (1003, '75367-1'), (1003, '10497-1'), (1003, '31117-1'),
        (1003, '60349-1'), (1003, '60351-1'), (1003, '60350-1'), (1003, '10340-1'),

        -- Lord Business (1004) — 5 sets
        (1004, '10276-1'), (1004, '75192-1'), (1004, '10294-1'), (1004, '10307-1'),
        (1004, '71043-1'),

        -- Kai Firefang (1005) — 10 sets
        (1005, '71767-1'), (1005, '71774-1'), (1005, '71741-1'), (1005, '71799-1'),
        (1005, '70751-1'), (1005, '76261-1'), (1005, '76178-1'), (1005, '76218-1'),
        (1005, '42143-1'), (1005, '42151-1'),

        -- Jay Walker (1006) — 3 sets
        (1006, '10281-1'), (1006, '21327-1'), (1006, '40460-1'),

        -- Lloyd Garmadon (1007) — 8 sets
        (1007, '21054-1'), (1007, '21044-1'), (1007, '21042-1'), (1007, '21034-1'),
        (1007, '60380-1'), (1007, '60364-1'), (1007, '60372-1'), (1007, '10278-1'),

        -- Nya Waterfall (1008) — 14 sets
        (1008, '42115-1'), (1008, '42143-1'), (1008, '42141-1'), (1008, '42145-1'),
        (1008, '42096-1'), (1008, '42110-1'), (1008, '42083-1'), (1008, '42056-1'),
        (1008, '42151-1'), (1008, '42154-1'), (1008, '42111-1'), (1008, '42126-1'),
        (1008, '42125-1'), (1008, '42127-1'),

        -- Cole Brookstone (1009) — 2 sets
        (1009, '10281-1'), (1009, '31120-1'),

        -- Zane Julien (1010) — 11 sets
        (1010, '71043-1'), (1010, '76405-1'), (1010, '75978-1'), (1010, '76391-1'),
        (1010, '76419-1'), (1010, '76389-1'), (1010, '76388-1'), (1010, '76395-1'),
        (1010, '76386-1'), (1010, '76387-1'), (1010, '76402-1'),

        -- Sensei Wu (1011) — 0 sets (intentionally absent)

        -- Laval Lionheart (1012) — 7 sets
        (1012, '10305-1'), (1012, '10332-1'), (1012, '21325-1'), (1012, '31120-1'),
        (1012, '10316-1'), (1012, '21348-1'), (1012, '10698-1'),

        -- Eris Eagleton (1013) — 4 sets
        (1013, '10280-1'), (1013, '10311-1'), (1013, '10313-1'), (1013, '10289-1'),

        -- Chase McCain (1014) — 16 sets
        (1014, '60380-1'), (1014, '60364-1'), (1014, '60372-1'), (1014, '10278-1'),
        (1014, '60349-1'), (1014, '60337-1'), (1014, '60336-1'), (1014, '60335-1'),
        (1014, '60327-1'), (1014, '60324-1'), (1014, '60316-1'), (1014, '60315-1'),
        (1014, '60292-1'), (1014, '60271-1'), (1014, '60197-1'), (1014, '60198-1'),

        -- Rex Dangervest (1015) — 6 sets
        (1015, '10300-1'), (1015, '21336-1'), (1015, '21330-1'), (1015, '10274-1'),
        (1015, '21319-1'), (1015, '21328-1'),

        -- Unikitty Cloudcuckoo (1016) — 9 sets
        (1016, '10280-1'), (1016, '10281-1'), (1016, '10311-1'), (1016, '10313-1'),
        (1016, '41703-1'), (1016, '43222-1'), (1016, '43197-1'), (1016, '10698-1'),
        (1016, '11717-1'),

        -- Metalbeard Pirate (1017) — 5 sets
        (1017, '21322-1'), (1017, '10320-1'), (1017, '31109-1'), (1017, '21316-1'),
        (1017, '10305-1'),

        -- Clutch Powers (1018) — 18 sets
        (1018, '75192-1'), (1018, '10276-1'), (1018, '10294-1'), (1018, '42115-1'),
        (1018, '71043-1'), (1018, '10307-1'), (1018, '21054-1'), (1018, '21309-1'),
        (1018, '10305-1'), (1018, '10255-1'), (1018, '21325-1'), (1018, '76178-1'),
        (1018, '42143-1'), (1018, '10312-1'), (1018, '10300-1'), (1018, '10316-1'),
        (1018, '43222-1'), (1018, '10283-1')
) AS ownership(Member_ID, Set_ID)